# Многокритериальная оптимизация параметров эпидемиологической модели SIR

**Цель:** Найти оптимальные комбинации параметров, минимизирующие одновременно
два критерия: пиковую заболеваемость и долю умерших. Используется эволюционный
алгоритм Borg MOEA из пакета BlackBoxOptim.

## Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"
using BlackBoxOptim, Random, Statistics

Подключение модуля с определением модели SIR

In [ ]:
include(srcdir("sir_model.jl"))

## Целевая функция для оптимизации

Функция `cost_multi` принимает вектор параметров и возвращает два критерия,
которые необходимо минимизировать:
1. Пиковая доля инфицированных
2. Доля умерших в популяции

### Параметры оптимизации

Вектор `x` имеет три компонента:
- `x[1]`: β_und — коэффициент заражения невыявленными больными (диапазон 0.1–1.0)
- `x[2]`: detection_time — время до выявления заболевания (диапазон 3–14 дней)
- `x[3]`: death_rate — вероятность летального исхода (диапазон 0.01–0.1)

In [ ]:
function cost_multi(x)

x[1]: β_und, x[2]: detection_time, x[3]: death_rate

### Первоначальная инициализация модели

Создаём модель с пробными параметрами для оценки пика заболеваемости

In [ ]:
    model = initialize_sir(;
        Ns = [1000, 1000, 1000],                    # три города по 1000 жителей
        β_und = fill(x[1], 3),                      # заразность невыявленных (одинакова для всех городов)
        β_det = fill(x[1]/10, 3),                   # заразность выявленных (в 10 раз ниже)
        infection_period = 14,                      # длительность болезни (дней)
        detection_time = round(Int, x[2]),          # время до выявления (целое число дней)
        death_rate = x[3],                          # вероятность смерти при завершении болезни
        reinfection_probability = 0.1,              # вероятность повторного заражения
        Is = [0, 0, 1],                             # начальные заражённые (только в третьем городе)
        seed = 42,                                  # зерно случайных чисел
        n_steps = 100,                              # длительность симуляции (дней)
    )

### Вспомогательные функции

- `infected_frac` — доля инфицированных в популяции
- `dead_count` — количество умерших (общая численность минус живые агенты)

In [ ]:
    infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    dead_count(model) = 3000 - nagents(model)       # 3000 = сумма Ns

### Множественные прогоны для учёта стохастичности

Запускаем 5 повторных симуляций с разными случайными зёрнами
для получения статистически значимых результатов

In [ ]:
    replicates = 5
    peak_vals = Float64[]    # массив для хранения пиковых значений заболеваемости
    dead_vals = Int[]        # массив для хранения количества умерших

    for rep = 1:replicates

Создаём модель с новым зерном для каждого прогона

In [ ]:
        model = initialize_sir(;
            Ns = [1000, 1000, 1000],
            β_und = fill(x[1], 3),
            β_det = fill(x[1]/10, 3),
            infection_period = 14,
            detection_time = round(Int, x[2]),
            death_rate = x[3],
            reinfection_probability = 0.1,
            Is = [0, 0, 1],
            seed = 42 + rep,                       # увеличиваем зерно для разнообразия
            n_steps = 100,
        )

### Выполнение симуляции

Запускаем модель на 100 шагов (дней) с помощью стандартного шага Agents.jl

In [ ]:
        for step = 1:100
            Agents.step!(model, 1)
            frac = infected_frac(model)
            if frac > peak_infected
                peak_infected = frac
            end
        end

Сохраняем результаты текущего прогона

In [ ]:
        push!(peak_vals, peak_infected)
        push!(dead_vals, dead_count(model))
    end

### Возврат критериев оптимизации

Возвращаем средние значения по всем прогонам:
- средняя пиковая заболеваемость
- средняя доля умерших (нормированная на общую численность)

In [ ]:
    return (mean(peak_vals), mean(dead_vals) / 3000)
end

## Запуск многокритериальной оптимизации

Используем алгоритм Borg MOEA (Multi-Objective Evolutionary Algorithm)
для поиска оптимальных параметров.

In [ ]:
result = bboptimize(
    cost_multi,
    Method = :borg_moea,                                    # метод оптимизации
    FitnessScheme = ParetoFitnessScheme{2}(is_minimizing = true),  # два критерия на минимизацию
    SearchRange = [
        (0.1, 1.0),      # β_und — коэффициент заражения
        (3.0, 14.0),     # detection_time — время до выявления (дни)
        (0.01, 0.1),     # death_rate — вероятность смерти
    ],
    NumDimensions = 3,                                       # количество оптимизируемых параметров
    MaxTime = 120,                                           # максимальное время выполнения (120 секунд = 2 минуты)
    TraceMode = :compact,                                    # компактный вывод прогресса
)

## Извлечение результатов оптимизации

In [ ]:
best = best_candidate(result)      # оптимальный вектор параметров
fitness = best_fitness(result)     # соответствующие значения критериев

## Вывод оптимальных параметров

In [ ]:
println("Оптимальные параметры:")
println("β_und = $(best[1])")
println("Время выявления = $(round(Int, best[2])) дней")
println("Смертность = $(best[3])")

## Вывод достигнутых показателей

In [ ]:
println("Достигнутые показатели:")
println("Пик заболеваемости: $(fitness[1])")
println("Доля умерших: $(fitness[2])")

## Сохранение результатов

Сохраняем оптимальные параметры и соответствующие им значения критериев
в JLD2-файл для последующего анализа.

In [ ]:
save(datadir("optimization_result.jld2"), Dict("best" => best, "fitness" => fitness))

## Интерпретация результатов

Результаты оптимизации позволяют найти компромиссные стратегии управления эпидемией:

- **Раннее выявление** (малое detection_time) снижает пик заболеваемости,
  но может не влиять на смертность
- **Низкая заразность** (β_und) напрямую влияет на оба критерия
- **Высокая смертность** может парадоксально снижать пик заболеваемости
  за счёт быстрого удаления инфицированных из популяции

Парето-фронт, найденный алгоритмом, показывает множество оптимальных
компромиссных решений, из которых лицо, принимающее решения, может
выбрать наиболее подходящее с учётом приоритетов.